In [5]:
# pip install transformers>=4.44 torch --upgrade
# Optional extras for extraction: pip install python-docx pdfplumber
from transformers import pipeline
import re
from collections import defaultdict

# 1) Load a PII model (swap to betterdataai/PII_DETECTION_MODEL if you like)
MODEL_ID = "iiiorg/piiranha-v1-detect-personal-information"  # :contentReference[oaicite:3]{index=3}
pii_pipe = pipeline("token-classification", model=MODEL_ID, aggregation_strategy="simple")

def read_policy_txt(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

# Optional: simple regex safety nets for formats that NER models sometimes miss
REGEX_PATTERNS = {
    "EMAIL": r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
    "IP_ADDRESS": r"\b(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)\b",
    "CREDIT_CARD": r"\b(?:\d[ -]*?){13,19}\b",
    "SSN": r"\b\d{3}-\d{2}-\d{4}\b",
    "PHONE_NUMBER": r"\+?\d[\d\-\s()]{7,}\d",
}

def detect_regex(text):
    hits = []
    for label, pat in REGEX_PATTERNS.items():
        for m in re.finditer(pat, text):
            hits.append({
                "entity_group": label,
                "start": m.start(),
                "end": m.end(),
                "score": 0.99,
                "word": text[m.start():m.end()]
            })
    return hits

def detect_pii(text: str, score_threshold: float = 0.50, max_chunk_chars: int = 3500):
    """
    Chunk long text for the model, merge results, and add regex-based detections.
    Returns (detections, redacted_text, summary_by_label).
    """
    detections = []
    i = 0
    while i < len(text):
        chunk = text[i:i+max_chunk_chars]
        # Avoid splitting words in the middle for nicer spans
        if i + max_chunk_chars < len(text):
            cut = chunk.rfind(" ")
            if cut > 0: chunk = chunk[:cut]
        # Model pass
        model_out = pii_pipe(chunk)
        for ent in model_out:
            if ent.get("score", 1.0) >= score_threshold:
                ent = dict(ent)
                ent["start"] += i
                ent["end"] += i
                detections.append(ent)
        i += len(chunk)

    # Add regex safety nets
    detections += detect_regex(text)

    # Merge overlapping spans with same label
    detections.sort(key=lambda x: (x["start"], x["end"]))
    merged = []
    for d in detections:
        if not merged:
            merged.append(d)
            continue
        last = merged[-1]
        if d["start"] <= last["end"] and d["entity_group"] == last["entity_group"]:
            last["end"] = max(last["end"], d["end"])
            last["word"] = text[last["start"]:last["end"]]
            last["score"] = max(last.get("score", 0), d.get("score", 0))
        else:
            merged.append(d)

    # Build redacted text
    redacted = []
    cursor = 0
    for m in merged:
        redacted.append(text[cursor:m["start"]])
        tag = m["entity_group"].upper().replace(" ", "_")
        redacted.append(f"[REDACTED_{tag}]")
        cursor = m["end"]
    redacted.append(text[cursor:])
    redacted_text = "".join(redacted)

    # Summary grouped by label
    by_label = defaultdict(list)
    for m in merged:
        snippet = text[max(0, m["start"]-40):min(len(text), m["end"]+40)]
        by_label[m["entity_group"]].append({
            "value": m["word"],
            "start": m["start"],
            "end": m["end"],
            "score": round(float(m.get("score", 0)), 3),
            "context": snippet
        })

    return merged, redacted_text, dict(by_label)



Device set to use cpu


In [ ]:
import os
ROOT_PATH = r"data/policies"

for root, _, files in os.walk(ROOT_PATH):
    for file in files:
        if file.lower().endswith(".txt"):
            file_path = os.path.join(root, file)
            print("\n" + "="*90)
            print(f"🔍 Scanning: {file_path}")
            print("="*90)

            txt = read_policy_txt(file_path)
            spans, redacted, summary = detect_pii(txt, score_threshold=0.50)

            print("\n=== PII Removal Checklist ===")
            if not summary:
                print("✅ No PII detected!")
            else:
                for label, items in summary.items():
                    print(f"- {label}: {len(items)} occurrence(s)")

                # Show a few examples
                for label, items in summary.items():
                    print(f"\n[{label}]")
                    for it in items[:5]:
                        print(f"  → '{it['value']}' @({it['start']}-{it['end']}) score={it['score']}")
                        print(f"   Context: ...{it['context']}...")


                # print("\n--- REDACTED TEXT PREVIEW ---")
                # print(redacted[:1500])  # only show first 1500 chars for readability
                # print("\n(Preview truncated)\n")


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



🔍 Scanning: Policies_Final\City of Rye\Rye_NY_Acceptable_Use_Policy_v2.txt

=== PII Removal Checklist ===
- IP_ADDRESS: 1 occurrence(s)

[IP_ADDRESS]
  → '3.5.11.1' @(6404-6412) score=0.99
   Context: ...better enhances business productivity. 
3.5.11.1 The IT department must test the request...

🔍 Scanning: Policies_Final\City of Rye\Rye_NY_Asset_Management_Policy_v1.0.txt

=== PII Removal Checklist ===
✅ No PII detected!

🔍 Scanning: Policies_Final\City of Rye\Rye_NY_Firewalls_and_Intrusion_Detection_Policy_v1.0.txt

=== PII Removal Checklist ===
✅ No PII detected!

🔍 Scanning: Policies_Final\City of Rye\Rye_NY_Information_Technology_Policy_v3.0.txt

=== PII Removal Checklist ===
- PASSWORD: 1 occurrence(s)

[PASSWORD]
  → ',!@#$%^&*)' @(2116-2126) score=0.967
   Context: ...s, numbers and special characters (i.e. ,!@#$%^&*)
* Passwords must not include your name ...

🔍 Scanning: Policies_Final\City of Rye\Rye_NY_Patch_and_Vulnerability_Management_Policy_v1.0.txt

=== PII Removal Che